# Anomaly Detection — Point Level

Scores every reading against the station's own recent behaviour rather than
against a global threshold, because a normal PM2.5 level in Liverpool is not a
normal level in Randwick.

| Detector | Question it answers |
|---|---|
| Rolling z-score | Is this reading far from this station's recent norm? |
| Flatline | Has the sensor reported the same value for hours? |
| Sensor health | Which station-months look unreliable? |

All SQL window functions. No ML libraries, no quota risk.

| Output | Grain |
|---|---|
| `workspace.aq_gold.observation_anomaly` | site × parameter × hour |
| `workspace.aq_gold.sensor_flatline` | one row per flatline run |
| `workspace.aq_gold.sensor_health` | site × parameter × month |

In [0]:
from pyspark.sql import functions as F

## 1. Rolling z-score

A z-score says how many standard deviations a value sits from the mean. The
window is the previous 168 hours (7 days) at the **same station** for the **same
pollutant**, excluding the current reading.

**Why exclude the current reading.** If a spike is included in its own baseline it
inflates the standard deviation and partially hides itself. `ROWS BETWEEN 168
PRECEDING AND 1 PRECEDING` stops one row short of the present.

**Why a rolling window rather than a fixed threshold.** "PM2.5 above 50 is an
anomaly" fails twice: it flags every reading at a genuinely polluted station, and
it misses a quiet station doubling from 5 to 10, which for that station is
remarkable.

| Threshold | Class |
|---|---|
| abs(z) > 5 | extreme |
| abs(z) > 3 | unusual |
| otherwise | normal |

Uses `value_clean` — the floored column — so below-detection readings do not
distort the baseline.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.observation_anomaly AS
WITH scored AS (
  SELECT
    site_id,
    parameter_code,
    obs_time,
    obs_date,
    obs_hour,
    value_clean,
    AVG(value_clean)    OVER w AS roll_mean,
    STDDEV(value_clean) OVER w AS roll_sd,
    LAG(value_clean, 1) OVER (PARTITION BY site_id, parameter_code ORDER BY obs_time)
                             AS prev_value
  FROM workspace.aq_silver.fact_observation
  WINDOW w AS (
    PARTITION BY site_id, parameter_code
    ORDER BY obs_time
    ROWS BETWEEN 168 PRECEDING AND 1 PRECEDING
  )
)
SELECT
  site_id,
  parameter_code,
  obs_time,
  obs_date,
  obs_hour,
  value_clean,
  ROUND(roll_mean, 3) AS roll_mean,
  ROUND(roll_sd, 3)   AS roll_sd,
  ROUND((value_clean - roll_mean) / NULLIF(roll_sd, 0), 2) AS z_score,
  ROUND(value_clean - prev_value, 3) AS hourly_delta,
  CASE
    WHEN roll_sd IS NULL OR roll_sd = 0 THEN 'insufficient_history'
    WHEN ABS((value_clean - roll_mean) / roll_sd) > 5 THEN 'extreme'
    WHEN ABS((value_clean - roll_mean) / roll_sd) > 3 THEN 'unusual'
    ELSE 'normal'
  END AS anomaly_class
FROM scored
""")

spark.sql("""
SELECT anomaly_class,
       count(*) AS rows,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM workspace.aq_gold.observation_anomaly
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------------+-------+-----+
|       anomaly_class|   rows|  pct|
+--------------------+-------+-----+
|              normal|3456427|98.56|
|             unusual|  43317| 1.24|
|             extreme|   7125| 0.20|
|insufficient_history|    160| 0.00|
+--------------------+-------+-----+



### CHECKPOINT — read the distribution above

| What you see | Meaning |
|---|---|
| `normal` above 95%, `unusual` a few %, `extreme` a fraction of a % | Working as intended |
| `extreme` at 10% or more | The threshold is wrong, not the data |
| `insufficient_history` large | Expected at the start of each station's series and after long gaps |

`NULLIF(roll_sd, 0)` prevents division by zero when a sensor has been perfectly
flat — which is itself the next detector.

### Anomalies by pollutant and year

In [0]:
spark.sql("""
SELECT parameter_code,
       year(obs_date) AS yr,
       sum(CASE WHEN anomaly_class = 'extreme' THEN 1 ELSE 0 END) AS extreme,
       sum(CASE WHEN anomaly_class = 'unusual' THEN 1 ELSE 0 END) AS unusual
FROM workspace.aq_gold.observation_anomaly
GROUP BY 1, 2
ORDER BY 1, 2
""").show(30)

+--------------+----+-------+-------+
|parameter_code|  yr|extreme|unusual|
+--------------+----+-------+-------+
|           NO2|2020|    293|   1977|
|           NO2|2021|    197|   1932|
|           NO2|2022|    271|   2043|
|           NO2|2023|    245|   1815|
|           NO2|2024|    240|   1834|
|           NO2|2025|    285|   2146|
|         OZONE|2020|     38|    839|
|         OZONE|2021|     27|   1043|
|         OZONE|2022|    133|   1124|
|         OZONE|2023|     13|    852|
|         OZONE|2024|     17|    902|
|         OZONE|2025|     89|   1195|
|          PM10|2020|    558|   2056|
|          PM10|2021|    566|   1988|
|          PM10|2022|    364|   1966|
|          PM10|2023|    580|   2234|
|          PM10|2024|    468|   2053|
|          PM10|2025|    530|   2190|
|         PM2.5|2020|    368|   2182|
|         PM2.5|2021|    395|   2078|
|         PM2.5|2022|    302|   2206|
|         PM2.5|2023|    420|   2176|
|         PM2.5|2024|    316|   2263|
|         PM

## 2. Flatline detection

A working air quality sensor essentially never reports the identical value for
many consecutive hours. When it does, the instrument has usually failed or is
repeating a held last-known value.

### The gaps-and-islands pattern

Two row numbers over the same rows — one ordered by time, one ordered by time
within each value. Their difference is constant across a consecutive run:

| obs_time | value | rn_all | rn_by_value | difference |
|---|---|---|---|---|
| 01:00 | 4.1 | 1 | 1 | 0 |
| 02:00 | 4.1 | 2 | 2 | 0 |
| 03:00 | 4.1 | 3 | 3 | 0 |
| 04:00 | 7.8 | 4 | 1 | 3 |
| 05:00 | 4.1 | 5 | 4 | 1 |

Rows sharing a difference form one run of identical values. Same trick used for
consecutive exceedance days in the gold layer.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.sensor_flatline AS
WITH runs AS (
  SELECT
    site_id, parameter_code, obs_time, value_clean,
    ROW_NUMBER() OVER (PARTITION BY site_id, parameter_code ORDER BY obs_time)
      - ROW_NUMBER() OVER (PARTITION BY site_id, parameter_code, value_clean ORDER BY obs_time)
      AS run_group
  FROM workspace.aq_silver.fact_observation
)
SELECT
  site_id,
  parameter_code,
  value_clean   AS stuck_value,
  MIN(obs_time) AS run_start,
  MAX(obs_time) AS run_end,
  COUNT(*)      AS hours_stuck
FROM runs
GROUP BY site_id, parameter_code, value_clean, run_group
HAVING COUNT(*) >= 6
""")

spark.sql("""
SELECT count(*) AS flatline_runs,
       max(hours_stuck) AS longest_run_hours,
       count(DISTINCT site_id) AS stations_affected
FROM workspace.aq_gold.sensor_flatline
""").show()

+-------------+-----------------+-----------------+
|flatline_runs|longest_run_hours|stations_affected|
+-------------+-----------------+-----------------+
|         4287|               91|               19|
+-------------+-----------------+-----------------+



In [0]:
spark.sql("""
SELECT s.site_name, f.parameter_code, f.stuck_value, f.hours_stuck, f.run_start
FROM workspace.aq_gold.sensor_flatline f
JOIN workspace.aq_silver.dim_station s USING (site_id)
ORDER BY f.hours_stuck DESC
LIMIT 15
""").show(truncate=False)

+----------------+--------------+-----------+-----------+-------------------+
|site_name       |parameter_code|stuck_value|hours_stuck|run_start          |
+----------------+--------------+-----------+-----------+-------------------+
|Randwick        |NO2           |0.0        |91         |2020-12-10 11:00:00|
|Randwick        |NO2           |0.0        |64         |2021-03-18 19:00:00|
|Randwick        |NO2           |0.0        |64         |2020-02-07 05:00:00|
|Rozelle         |NO2           |0.0        |61         |2022-01-05 20:00:00|
|Bringelly       |NO2           |0.0        |58         |2020-12-31 06:00:00|
|Cook And Phillip|PM2.5         |0.0        |58         |2021-07-24 11:00:00|
|Rozelle         |NO2           |0.0        |58         |2022-01-28 14:00:00|
|Richmond        |NO2           |0.0        |57         |2022-10-31 09:00:00|
|Macquarie Park  |NO2           |0.0        |56         |2022-01-05 17:00:00|
|Richmond        |NO2           |0.0        |49         |2023-04

### CHECKPOINT — inspect these by eye

**Expect false positives, and check for them.**

Because `value_clean` floors negatives at zero, every consecutive run of
below-detection readings becomes a run of exact zeros — which looks identical to a
dead sensor. PM2.5 is 9.14% below detection, so this will happen.

| What the top results show | Interpretation |
|---|---|
| Mostly `stuck_value = 0.0` | Below-detection runs, not faults. Exclude zeros or report separately |
| Long runs at a non-zero value | Genuine flatline. A real finding |

This is a data quality rule that needs auditing before it is trusted — the same
lesson as the negative-value finding.

In [0]:
spark.sql("""
SELECT
  CASE WHEN stuck_value = 0 THEN 'zero (below detection)' ELSE 'non-zero (likely fault)' END AS run_type,
  count(*) AS runs,
  max(hours_stuck) AS longest
FROM workspace.aq_gold.sensor_flatline
GROUP BY 1 ORDER BY 2 DESC
""").show(truncate=False)

+----------------------+----+-------+
|run_type              |runs|longest|
+----------------------+----+-------+
|zero (below detection)|4287|91     |
+----------------------+----+-------+



## 3. Sensor health summary

Rolls the hourly signals up to a monthly report card per station per parameter.
This is the table an operations team would actually look at.

`DAY(LAST_DAY(month_start)) * 24` computes how many hourly readings that month
should have contained, handling different month lengths and leap years without a
lookup table.

| Status | Assigned when |
|---|---|
| `poor_coverage` | Under 50% of expected readings present |
| `suspect` | Over 2% of readings classed extreme |
| `gappy` | Under 90% of expected readings present |
| `healthy` | Everything else |

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.sensor_health AS
WITH monthly AS (
  SELECT
    a.site_id,
    a.parameter_code,
    DATE_TRUNC('MONTH', a.obs_date) AS month_start,
    COUNT(*)                                                     AS readings,
    SUM(CASE WHEN a.anomaly_class = 'extreme' THEN 1 ELSE 0 END) AS extreme_count,
    SUM(CASE WHEN a.anomaly_class = 'unusual' THEN 1 ELSE 0 END) AS unusual_count,
    ROUND(AVG(a.value_clean), 3)                                 AS mean_value,
    ROUND(STDDEV(a.value_clean), 3)                              AS sd_value
  FROM workspace.aq_gold.observation_anomaly a
  GROUP BY 1, 2, 3
),
expected AS (
  SELECT *, DAY(LAST_DAY(month_start)) * 24 AS expected_readings
  FROM monthly
)
SELECT
  e.*,
  ROUND(100.0 * e.readings / e.expected_readings, 1) AS completeness_pct,
  ROUND(100.0 * e.extreme_count / e.readings, 2)     AS extreme_pct,
  CASE
    WHEN 100.0 * e.readings / e.expected_readings < 50 THEN 'poor_coverage'
    WHEN 100.0 * e.extreme_count / e.readings     > 2  THEN 'suspect'
    WHEN 100.0 * e.readings / e.expected_readings < 90 THEN 'gappy'
    ELSE 'healthy'
  END AS health_status
FROM expected e
""")

spark.sql("""
SELECT health_status, count(*) AS station_months,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM workspace.aq_gold.sensor_health
GROUP BY 1 ORDER BY 2 DESC
""").show()

+-------------+--------------+----+
|health_status|station_months| pct|
+-------------+--------------+----+
|      healthy|          4476|88.1|
|        gappy|           555|10.9|
|poor_coverage|            40| 0.8|
|      suspect|             8| 0.2|
+-------------+--------------+----+



### Worst station-months

In [0]:
spark.sql("""
SELECT s.site_name, h.parameter_code, h.month_start,
       h.completeness_pct, h.extreme_pct, h.health_status
FROM workspace.aq_gold.sensor_health h
JOIN workspace.aq_silver.dim_station s USING (site_id)
WHERE h.health_status <> 'healthy'
ORDER BY h.completeness_pct ASC
LIMIT 20
""").show(truncate=False)

+----------------+--------------+-------------------+----------------+-----------+-------------+
|site_name       |parameter_code|month_start        |completeness_pct|extreme_pct|health_status|
+----------------+--------------+-------------------+----------------+-----------+-------------+
|Richmond        |PM2.5         |2024-02-01 00:00:00|3.6             |4.00       |poor_coverage|
|Rouse Hill      |PM2.5         |2024-02-01 00:00:00|3.6             |0.00       |poor_coverage|
|Cook And Phillip|PM2.5         |2021-08-01 00:00:00|4.2             |0.00       |poor_coverage|
|Cook And Phillip|PM10          |2021-08-01 00:00:00|5.8             |0.00       |poor_coverage|
|Prospect        |PM2.5         |2025-06-01 00:00:00|11.4            |4.88       |poor_coverage|
|Prospect        |PM2.5         |2025-09-01 00:00:00|14.0            |0.00       |poor_coverage|
|Lidcombe        |NO2           |2020-03-01 00:00:00|15.6            |0.00       |poor_coverage|
|Lidcombe        |OZONE       

## 4. Prove the grain

Both new gold tables must have unique keys. Zero means the grain holds.

In [0]:
checks = [
    ("observation_anomaly", "site_id, parameter_code, obs_time"),
    ("sensor_health",       "site_id, parameter_code, month_start"),
]

for table, keys in checks:
    n = spark.sql(f"""
        SELECT {keys}, count(*) AS c
        FROM workspace.aq_gold.{table}
        GROUP BY {keys} HAVING count(*) > 1
    """).count()
    print(f"{table:22} grain violations: {n}")

observation_anomaly    grain violations: 0
sensor_health          grain violations: 0


## 5. First look at January 2020

Not the formal validation — that is tomorrow's notebook. This is a sanity check
that the detector responds at all during the Black Summer smoke period.

**Watch for baseline adaptation.** The rolling window is 7 days. If smoke
persisted for weeks, the baseline learns it and the detector goes quiet in the
middle of the worst air. If `mean_pm25` stays high while `anomalies` drops to
zero after a few days, that is exactly what happened — and it is a finding worth
writing up, not a failure.

In [0]:
spark.sql("""
SELECT DATE_TRUNC('DAY', obs_time) AS day,
       ROUND(AVG(value_clean), 1) AS mean_pm25,
       SUM(CASE WHEN anomaly_class IN ('unusual','extreme') THEN 1 ELSE 0 END) AS anomalies,
       COUNT(DISTINCT site_id) AS stations
FROM workspace.aq_gold.observation_anomaly
WHERE parameter_code = 'PM2.5'
  AND obs_time >= '2020-01-01' AND obs_time < '2020-02-01'
GROUP BY 1 ORDER BY 1
""").show(35, truncate=False)

+-------------------+---------+---------+--------+
|day                |mean_pm25|anomalies|stations|
+-------------------+---------+---------+--------+
|2020-01-01 00:00:00|28.4     |37       |14      |
|2020-01-02 00:00:00|29.0     |10       |15      |
|2020-01-03 00:00:00|21.4     |1        |14      |
|2020-01-04 00:00:00|32.9     |27       |14      |
|2020-01-05 00:00:00|66.6     |45       |13      |
|2020-01-06 00:00:00|11.9     |0        |14      |
|2020-01-07 00:00:00|17.2     |0        |14      |
|2020-01-08 00:00:00|77.2     |64       |15      |
|2020-01-09 00:00:00|18.0     |0        |15      |
|2020-01-10 00:00:00|13.5     |0        |15      |
|2020-01-11 00:00:00|38.7     |6        |15      |
|2020-01-12 00:00:00|46.1     |0        |15      |
|2020-01-13 00:00:00|23.0     |0        |15      |
|2020-01-14 00:00:00|7.6      |0        |15      |
|2020-01-15 00:00:00|6.6      |0        |15      |
|2020-01-16 00:00:00|17.8     |0        |15      |
|2020-01-17 00:00:00|32.4     |

## 6. Table comments — feeds Genie

In [0]:
spark.sql("""
COMMENT ON TABLE workspace.aq_gold.observation_anomaly IS
  'One row per station, per pollutant, per hour. Each reading scored against a rolling 7-day baseline for that same station and pollutant, and classified normal, unusual or extreme.'
""")

spark.sql("""
ALTER TABLE workspace.aq_gold.observation_anomaly
  ALTER COLUMN z_score COMMENT 'Standard deviations from the rolling 7-day mean for this station and pollutant. The current reading is excluded from its own baseline'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.sensor_health IS
  'One row per station, per pollutant, per month. Completeness against expected hourly readings, extreme-anomaly rate, and an overall health status.'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.sensor_flatline IS
  'One row per run of six or more consecutive identical readings at a station. Runs at zero reflect below-detection periods rather than instrument faults.'
""")

print("comments applied")

comments applied
